# Load and process Transparent California Data

### Prerequisite

Download the data.zip folder from the google drive and extract it in the project directory.

Also download the csu_combined.parquet and SalaryRecsCSU.csv into the data/raw/ directory

The directory structure for this pipline should look like the following

```md
data/
    raw/
    processed/
data-pipline
    notebooks.ipynb
```

In [119]:
import os

# Check or create prerequisite folders
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/raw", exist_ok=True)



In [120]:
# Lossless data processing for provided downloaded from Transparent Califronia

import polars as pl

print("Scanning CSV files...")

# Numerical cols
financial_cols = [
    "base", "overtime", "other", "totalpay", "totalbenefits",
    "totalpaybenefits"
]

df = pl.scan_csv(
    "../data/raw/tc_raw/**/*.csv",
    include_file_paths="filepath"
)
# Empty Cols
df = df.drop(['ual', 'benefitsee', 'benefitser', 'benefitsdc'], strict=False)

# Helper function to currency text into Ints!
def clean_currency(col_name):
    return ( pl.col(col_name) .cast(pl.Int32, strict=False))

df = df.rename(str.lower)

# Apply typing and data transformations
df = df.with_columns(
    pl.selectors.string().str.to_lowercase()
).with_columns(
    # Get Year From Filename
    pl.col("filepath").str.extract(r"-(\d{4})\.csv$", 1).cast(pl.Int16).alias("year"),

    # Categorical cols
    pl.col("job").cast(pl.Categorical),
    pl.col("department").cast(pl.Categorical),

    # Clean and cast all financial columns
    *[clean_currency(col) for col in financial_cols]
)

df = df.drop(["filepath"])

df = df.drop_nulls(subset=["name"])

df.describe()

Scanning CSV files...


statistic,name,job,department,base,overtime,other,benefitshw,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status
str,str,str,str,f64,f64,f64,str,f64,f64,f64,str,f64,str,str
"""count""","""987543""","""987543""","""0""",987543.0,987543.0,987543.0,"""0""",987543.0,987543.0,987543.0,"""28458""",987543.0,"""987543""","""0"""
"""null_count""","""0""","""0""","""987543""",0.0,0.0,0.0,"""987543""",0.0,0.0,0.0,"""959085""",0.0,"""0""","""987543"""
"""mean""",null,null,null,38954.754078,251.729386,2990.511412,null,42197.157741,18286.612329,60484.049206,null,2019.001874,null,null
"""std""",null,null,null,41626.826402,2194.793662,7709.038972,null,44443.447239,20332.057299,63379.724403,null,3.35553,null,null
"""min""",""" judiyaba""",null,null,-5669.0,-72.0,-23732.0,null,0.0,-95066.0,-64657.0,"""employee held multiple positio…",2013.0,"""california state university""",null
"""25%""",null,null,null,412.0,0.0,0.0,null,2430.0,0.0,2500.0,null,2016.0,null,null
"""50%""",null,null,null,29019.0,0.0,2.0,null,32434.0,11413.0,46481.0,null,2019.0,null,null
"""75%""",null,null,null,63965.0,0.0,2500.0,null,67690.0,33840.0,100952.0,null,2022.0,null,null
"""max""","""zvonimir hlousek""",null,null,790125.0,249767.0,473046.0,null,970004.0,181837.0,1.114304e6,"""employee holds multiple positi…",2024.0,"""california state university""",null


In [121]:
import re

import pandas as pd

CSU_ROLES = [
    ("supplemental_teaching_pay", r"special programs|summer session|extension|additional employment|music studio"),
    ("ladder_faculty", r"^instructional faculty|^grant-related/specially funded instructional faculty|^department chair|^cma instructional faculty"),
    ("lecturer", r"lecturer"),
    ("clinical_adjunct_faculty", r"^visiting faculty"),
    ("postdoc", r"^research fellow|post-doctoral"),
    ("trainee", r"counselor intern"),
    ("researcher_specialist_librarian", r"librarian|academic-related|coach"),
]


def role_of(title, rules) -> str:
    t = str(title or "").lower().strip()
    return next((role for role, pattern in rules if re.search(pattern, t)), "staff")

df = df.collect().to_pandas()

df = df.copy()
df["role"] = df["job"].map(lambda t: role_of(t, CSU_ROLES)).astype("string")

# Replace "name" with your actual name-column name if needed
df["name"] = (
    df["name"]
    .str.title()
    .str.replace(r"(?<=\s)[A-Z]\.?(?=\s|$)", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# The way we are droping duplicates should be evaluated

# num_dups = df.duplicated(subset=["year", "name", "job"]).sum()
# print(f"Number of duplicate rows: {num_dups}")

# dups_df = df[df.duplicated(subset=["year", "name", "job"], keep=False)]

# print(dups_df)

# counts = df.value_counts(subset=["year", "name", "job"])
# print(counts[counts > 1])


# Remove Duplicate Values
df = df.drop_duplicates(subset=["year", "name", "job"])

df = pl.from_pandas(df)

df.describe()

statistic,name,job,department,base,overtime,other,benefitshw,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status,role
str,str,str,f64,f64,f64,f64,str,f64,f64,f64,str,f64,str,str,str
"""count""","""943556""","""943556""",0.0,943556.0,943556.0,943556.0,"""0""",943556.0,943556.0,943556.0,"""28344""",943556.0,"""943556""","""0""","""943556"""
"""null_count""","""0""","""0""",943556.0,0.0,0.0,0.0,"""943556""",0.0,0.0,0.0,"""915212""",0.0,"""0""","""943556""","""0"""
"""mean""",null,null,null,40398.748916,262.691425,3082.202439,null,43743.810833,18964.355998,62708.455843,null,2018.927074,null,null,null
"""std""",null,null,null,41752.510475,2242.975016,7817.771994,null,44541.706162,20405.017431,63506.469725,null,3.368411,null,null,null
"""min""","""A Alisia Nantze""",null,null,-5669.0,-72.0,-23732.0,null,0.0,-95066.0,-64657.0,"""employee held multiple positio…",2013.0,"""california state university""",null,"""clinical_adjunct_faculty"""
"""25%""",null,null,null,2310.0,0.0,0.0,null,3800.0,0.0,3960.0,null,2016.0,null,null,null
"""50%""",null,null,null,31876.0,0.0,54.0,null,35102.0,14047.0,51391.0,null,2019.0,null,null,null
"""75%""",null,null,null,65517.0,0.0,2694.0,null,69390.0,34498.0,103235.0,null,2022.0,null,null,null
"""max""","""Zvonimir Hlousek""",null,null,790125.0,249767.0,473046.0,null,970004.0,181837.0,1.114304e6,"""employee holds multiple positi…",2024.0,"""california state university""",null,"""trainee"""


# Attach Transparent California Data to Scrapped CSU Data

Make sure to import "all_csu_scraped_data_combined.parquet" from Raw data files and "transparent_california_no_dup_filtered_csu_salaries.parquet" from /Faculty Data/CSU Data/ transparent_california_no_dup_filtered_csu_salaries.parquet" into the Colab before you start


In [122]:
import pandas as pd

def prepare_names(df):
    df = df.copy()
    names = df["name"].astype("string").str.strip()

    # Remove trailing suffixes before handling commas.
    names = names.str.replace(
        r"(?i)[,\s]+(?:jr|sr|ii|iii|iv)\.?$", "", regex=True
    )

    # Convert "Last, First" to "First Last" for any campus.
    names = names.str.replace(
        r"^([^,]+),\s*([^,]+)$", r"\2 \1", regex=True
    )

    names = names.str.replace(r"\s+", " ", regex=True).str.strip()
    df["name"] = names

    tokens = names.str.casefold().str.split()
    df["_first"] = tokens.str[0]
    df["_last"] = tokens.str[-1]

    return df.dropna(subset=["_first", "_last"])


csu_pay = df.to_pandas()
raw_csu = pd.read_parquet("../data/raw/csu_combined.parquet")

pay = prepare_names(csu_pay.loc[csu_pay["role"] != "staff"])
faculty = prepare_names(raw_csu)

joined = (
    pay.merge(
        faculty,
        on=["_first", "_last"],
        how="inner",
        suffixes=("_transparentCA", "_scraped"),
    )
    .drop(columns=["_first", "_last", "name_transparentCA"])
    .rename(columns={"name_scraped": "name"})
)

joined = joined[["name"] + [col for col in joined.columns if col != "name"]]
print(f"Matched rows: {len(joined):,}")
joined.head(10)


cleaned_columns = [
    "name", "job", "role", "base", "overtime", "other", "totalpay",
    "totalbenefits", "totalpaybenefits", "year_transparentCA",
    "title", "department_scraped", "school", "year_scraped",
    "major", "start_date",
]

cleaned_joined = joined.loc[:, cleaned_columns].copy()

print(f"Saved {len(cleaned_joined):,} cleaned joined rows")


Matched rows: 173,379
Saved 173,379 cleaned joined rows


In [123]:
raw_csu.describe()


,name,department,title,school,first_last_name,major,year,start_date
count,18040,17569,16803,18040,18040,495,1848,1494
unique,17384,1324,621,22,17395,19,80,47
top,University (Ph.D.,Psychology,Professor,CSU Long Beach,University (Ph.D.,Null,,2017
freq,16,668,4574,1843,16,460,370,93


<!-- DELETE THIS -->
### Load Data

In [124]:
import polars as pl
import duckdb

df = pl.from_pandas(cleaned_joined)

<!-- DELETE THIS -->
### Basic Data Information


<!-- DELETE THIS -->
### Role Column
No rows are dropped here. `role` comes from the Transparent California job title.
`filtered_faculty` is the stable input to department bucketing; `faculty_salaries` is the working table.

#### I am unsure about this sql statement

In [125]:
# Rebuild from the original input so this cell is safe to rerun.
df = duckdb.sql("SELECT DISTINCT * FROM df").pl()
duckdb.sql("CREATE OR REPLACE TABLE filtered_faculty AS SELECT * FROM df")
duckdb.sql("CREATE OR REPLACE TABLE faculty_salaries AS SELECT * FROM filtered_faculty")

<!-- DELETE THIS -->
Data info after adding roles

# Depatment Buckets

## Proposed Department Buckets
* Engineering
* Computer Science
* Science & Mathematics
* Medicine & Heath Sciences
* Business, Economics & Finance
* Agriculture, Food & Animal
* Architecture & Environmental Design
* Humanities & Languages
* Social Sciences & Communication
* Education & Public Administration

### Creating the Proposed Buckets
Rules use normalized department text and first-match precedence. Education rules precede the
humanities/social rules; `art`/`arts` must be complete words, and there is no generic `studies` rule.
Known subjects are mapped below. Missing or ambiguous departments stay in `Other` for review.
Each run rebuilds from `filtered_faculty`, avoiding stale or duplicated bucket columns.

In [126]:
bucket_sql = """
SELECT *, lower(trim(department_scraped)) as department,
CASE
    WHEN
        department ILIKE '%computer%'
        OR department ILIKE '%comp sci%'
        OR department ILIKE '%cmptr sci%'
        OR department ILIKE '%computing%'
        OR department ILIKE '%software%'
        OR department ILIKE '%informatics%'
        OR department ILIKE '%cyber%'
        OR department ILIKE '%data science%'
        OR department ILIKE '%information studies%'
        OR department ILIKE '%school of info%'
    THEN 'Computer Science'

--- Adding Catches Before we filter out all the Hospital People 
    WHEN
        department ILIKE '%bioengineering%'
    THEN 'Engineering'

    WHEN
        department ILIKE '%biochemistry and biophysics%'
        OR department ILIKE '%biochem & molecular med%'
        OR department ILIKE '%microbiology and immunology%'
        OR department ILIKE '%medical microbiology & imm%'
        OR department ILIKE '%cell and tissue biology%'
        OR department ILIKE '%cell biology & human anat%'
        OR department ILIKE '%anat physio & cell biology%'
        OR department ILIKE '%physiology & membrane biol%'
        OR department ILIKE '%plant pathology%'
        OR department ILIKE '%pharmaceutical chemistry%'
        OR department ILIKE '%pharmaceutical science%'
        OR department ILIKE '%cellular & molecular medicine%'
        OR department ILIKE '%cellular & molecular pharmacol%'
        OR department ILIKE '%molecular & medical pharmacol%'
        OR department ILIKE '%molecular bio sciences%'
        OR department ILIKE 'pharmacology'
        OR department ILIKE 'med:%pharmacology'
        OR department ILIKE 'm!_anatomy%' ESCAPE '!'
        OR department ILIKE 'm!_physiology' ESCAPE '!'
        OR department ILIKE 'm!_cmp (cellular molecular pha)' ESCAPE '!'
        OR department ILIKE 'som neuroscience'
        OR department ILIKE 'chemical sciences'
        OR department ILIKE 'chemistry & chemical biology'
    THEN 'Science & Mathematics'    

    WHEN
        department ILIKE '%medicine%'
        OR department ILIKE '%surgery%'
        OR department ILIKE '%vm:%'
        OR department ILIKE '%med:%'
        --OR (
        --    campus = 'san francisco'
        --    AND (
        --        department ILIKE 'm!_%' ESCAPE '!'
        --        OR department ILIKE 'n!_%' ESCAPE '!'
        --        OR department ILIKE 'd!_%' ESCAPE '!'
        --        OR department ILIKE 'p!_%' ESCAPE '!'
        --        OR department ILIKE 'h!_%' ESCAPE '!'
        --    )
        --)
        OR department ILIKE '%nurs%'
        OR department ILIKE '%dentistry%'
        OR department ILIKE '%pharma%'
        OR department ILIKE '%gynecology%'
        OR department ILIKE '%urology%'
        OR department ILIKE '%oncology%'
        OR department ILIKE '%family med%'
        OR department ILIKE '%radiation%'
        OR department ILIKE '%ophthalmology%'
        OR department ILIKE '%pathology%'
        OR department ILIKE '%peds%'
        OR department ILIKE '%dermatology%'
        OR department ILIKE '%pediatrics%'
        OR department ILIKE '%anesthesiology%'
        OR department ILIKE '%optometry%'
        OR department ILIKE '%psychiatry%'
        OR department ILIKE '%emerg med%'
        OR department ILIKE '%radiolog%'
        OR department ILIKE '%radiation%'
        OR department ILIKE '%anesthes%'
        OR department ILIKE '%public health%'
        OR department ILIKE '%med ed administration%'
        OR department ILIKE '%deans office-school of medicin%'
        OR department ILIKE '%semel institute%'
        OR department ILIKE '%school of med%'
        OR department ILIKE '%dom cardiology%'
        OR department ILIKE '%ophth clinical%'
        OR department ILIKE '%med-campus%'
        OR department ILIKE '%dntl sci%'
        OR department ILIKE '%ob/gyn%'
        OR department ILIKE '%ambulatory%'
        OR department ILIKE '%electronic health record%'
        OR department ILIKE '%electrocardiology%'
        OR department ILIKE '%epidemiology%'
        OR department ILIKE '%otolaryngology%'
        OR department ILIKE '%hematology%'
        OR department ILIKE '%school of public health%'
        OR department ILIKE '%health policy and management%'
        OR department ILIKE '%health sciences%'
        OR department ILIKE '%lppi: instr & research%'
        OR department ILIKE 'dom %'
        OR department ILIKE 'som %'
        OR department ILIKE 'som-%'
        OR department ILIKE '%obgyn%'
        OR department ILIKE '%sfgh%'
        OR department ILIKE 'psych-%'
        OR department ILIKE 'fcm-%'
        OR department ILIKE '%health science%'
        OR department ILIKE '%health care%'
        OR department ILIKE '%health serv%'
        OR department ILIKE '%pub health%'
        OR department ILIKE '%applied health%'
        OR department ILIKE '%kinesiology%'
        OR department ILIKE '%occupational therapy%'
        OR department ILIKE '%audiology%'
        OR department ILIKE '%clinical laboratory%'
        OR department ILIKE '%clinical science%'
        OR department ILIKE '%gerontology%'
        OR department ILIKE '%marital and family therapy%'
        OR department ILIKE '%neurology%'
        OR department ILIKE '%physical therapy%'
        OR department ILIKE '%physical med%'
        OR department ILIKE '%hospital%'
        OR department ILIKE '%electrophysiology%'
        OR department ILIKE '%physician%'
        OR department ILIKE '%medical ed%'
        OR department ILIKE '%dental%'
    THEN NULL

   WHEN
        department ILIKE 'phys ed program'
        OR department ILIKE 'physical education program'
    THEN 'Education'

    WHEN
        department ILIKE 'physical activities%'
    THEN 'Other'

    WHEN
        department ILIKE '%business%'
        OR department ILIKE '%econom%'
        OR department ILIKE 'econ%'
        OR department ILIKE '% econ%'
        OR department ILIKE '%finance%'
        OR department ILIKE '%of managemen%'
        OR department ILIKE '%haas core%'
        OR department ILIKE '%technology management program%'
        OR department ILIKE '%grad.sch. int.relat./pac.stud.%'
        OR (
            department ILIKE '%accounting%'
            AND department NOT ILIKE '%office%' -- dodges some non educators
        )
        OR department ILIKE '%accountancy%'
        OR department ILIKE '%marketing%'
        OR department ILIKE 'management'
        OR department ILIKE '%management and human resources%'
        OR department ILIKE '%management & human resources%'
    THEN 'Business, Economics & Finance'

    WHEN
        department ILIKE '%engineer%'
        OR department ILIKE '%engr%'
        OR department ILIKE '%mech%'
        OR department ILIKE '%civil%'
        OR department ILIKE '%areo%'
        OR department ILIKE '%electric%'
        OR department ILIKE '%soe-ams%'
        OR department ILIKE '%chemical%'
        OR department ILIKE '%industrial%'
        OR department ILIKE '%material%'
        OR department ILIKE '%soece-ee-tm-cm%'
    THEN 'Engineering'

    WHEN
        department ILIKE '%math%'
        OR department ILIKE '%stat%'
        OR department ILIKE '%statistics%'
        OR department ILIKE '%bio%'
        OR department ILIKE '%neuro%'
        OR department ILIKE '%phys%'
        OR department ILIKE '%land air & water resources%'
        OR department ILIKE '%climate/atmos sci/phy oceanog%'
        OR department ILIKE '%ocean sciences%'
        OR department ILIKE '%entomology%'
        OR department ILIKE '%physics%'
        OR department ILIKE '%astro%'
        OR department ILIKE '%chem%'
        OR department ILIKE '%sio dep%'
        OR department ILIKE '%molecular, cell%'
        OR department ILIKE '%earth%'
        OR department ILIKE '%oceanic%'
        OR department ILIKE '%ecology%'
        OR department ILIKE '%genetics%'
        OR department ILIKE '%igpp%'
        OR department ILIKE '%geosciences%'
        OR department ILIKE '%oceanography%'
        OR department ILIKE '%geology%'
        OR department ILIKE '%viticulture%'
        OR department ILIKE '%nematology%'
        OR department ILIKE '%demography%'
    THEN 'Science & Mathematics'

    WHEN
        department ILIKE '%plant science%'
        OR department ILIKE '%agriculture%'
        OR department ILIKE '%espm%'
        OR department ILIKE '%nutrition%'
        OR department ILIKE '%inst of environment and sustai%'
        OR department ILIKE '%energy & resources group%'
        OR department ILIKE '%animal%'
        OR department ILIKE '%food science%'
        OR department ILIKE '%food safety%'
    THEN 'Agriculture, Food & Animal'

    WHEN
        department ILIKE '%architec%'
        OR department ILIKE '%planning%'
        OR department ILIKE '%urban%'
        OR department ILIKE '%landscape arch%'
        OR department ILIKE '%environmental%'
    THEN 'Architecture & Environmental Design'

    WHEN
        department ILIKE 'law'
        OR department ILIKE 'law/%'
        OR department ILIKE 'law %'
        OR department ILIKE '%school of law%'
        OR department ILIKE '%law school%'
        OR department ILIKE '% law %'
        OR department ILIKE '% law'
        OR department ILIKE '%legal%'
        OR department ILIKE 'jd program'
    THEN 'Public Administration & Law'

    WHEN
        department ILIKE '%education%'
        OR department ILIKE '%educational%'
        OR department ILIKE '%teacher%'
        OR department ILIKE '%teaching%'
        OR department ILIKE '%edleadership%'
        OR department ILIKE '%ed doctorate%'
        OR department ILIKE '%undergrad ed%'
        OR department ILIKE '%extended ed%'
        OR department ILIKE '%learning design%'
        OR department ILIKE '%instructional technology%'
        OR department ILIKE '%lifelong learning%'
    THEN 'Education'

    WHEN
        department ILIKE '%public administration%'
        OR department ILIKE '%public policy%'
        OR department ILIKE '%public affairs%'
        OR department ILIKE '%pub pol%'
        OR department ILIKE '%policy&strategy%'
        OR department ILIKE '%homeland security%'
    THEN 'Public Administration & Law'

    WHEN
        department ILIKE '%language%'
        OR department ILIKE '%humanities%'
        OR department ILIKE '%english%'
        OR department ILIKE '%literature%'
        OR department ILIKE '%writing%'
        OR department ILIKE '%history%'
        OR department ILIKE '%philosophy%'
        OR department ILIKE '%french%'
        OR department ILIKE '%spanish%'
        OR department ILIKE '%german%'
        OR department ILIKE '%japanese%'
        OR department ILIKE '%dance%'
        OR department ILIKE '%music%'
        OR department ILIKE '%division of arts%'
        OR department ILIKE '%classics%'
        OR department ILIKE '%film%'
        OR department ILIKE '%visual arts%'
        OR department ILIKE '%theatre%'
        OR department ILIKE '%religious%'
        OR department ILIKE '%theater%'
        OR department ILIKE '%asian lang%'
        OR department ILIKE '%eastern lang%'
        OR department ILIKE '%world arts%'
        OR department ILIKE '%southeast asian%'
        OR department ILIKE '%drama department%'
        OR department ILIKE '%studio art%'
        OR department ILIKE 'art'
        OR department ILIKE 'arts'
        OR department ILIKE 'art %'
        OR department ILIKE 'arts %'
        OR department ILIKE 'arts-%'
        OR department ILIKE '% art'
        OR department ILIKE '% arts'
        OR department ILIKE '% art %'
        OR department ILIKE '% arts %'
        OR department ILIKE '%design program%'
        OR department ILIKE '%rhetoric%'
        OR department ILIKE '%near eastern%'
        OR department ILIKE '%performance, play & design%'
        OR department ILIKE '%drama%'
        OR department ILIKE '%hispanic%'
        OR department ILIKE '%italian%'
        OR department ILIKE '%scandinavian%'
        OR department ILIKE '%slavic%'
        OR department ILIKE '%greek & roman%'
        OR department ILIKE '%department of design%'
        OR department ILIKE '%media%'
        OR department ILIKE '%institute for society and gene%'
        OR department ILIKE '%college of creative studies%'    
        OR department ILIKE '%journalism%'
    THEN 'Humanities & Languages'

    WHEN
        department ILIKE '%anthropology%'
        OR department ILIKE '%linguistics%'
        OR department ILIKE '%department of chicana%'
        OR department ILIKE '%asian-american%'
        OR department ILIKE '%asian%'
        OR department ILIKE '%chicano%'
        OR department ILIKE '%american studies%'
        OR department ILIKE '%italian studies%'
        OR department ILIKE '%native american studies%'
        OR department ILIKE '%african%'
        OR department ILIKE '%Social Science%'
        OR department ILIKE '%sociology%'
        OR department ILIKE '%political science%'
        OR department ILIKE '%politics%'
        OR department ILIKE '%ethnic stud%'
        OR department ILIKE '%women%'
        OR department ILIKE '%psychology%'
        OR department ILIKE '%psychological%'
        OR department ILIKE '%cognitive science%'
        OR department ILIKE '%global & intl studies%'
        OR department ILIKE '%soc sci undergrad%'
        OR department ILIKE '%comm%'
        OR department ILIKE '%gender%'
        OR department ILIKE '%child%'
        OR department ILIKE '%geography%'
        OR department ILIKE '%human ecology%'
        OR department ILIKE '%social welfare%'
        OR department ILIKE '%global & international studies%'
        OR department ILIKE '%african am studies%'
        OR department ILIKE '%black studies%'
        OR department ILIKE '%studies%'
        OR department ILIKE '%science & technology studies%'    
        OR department ILIKE 'm!_anthro%' ESCAPE '!'   
    THEN 'Social Sciences & Communication'
    
    ELSE 'Other'
END AS dep_bucket
FROM faculty_salaries
WHERE dep_bucket IS NOT NULL
"""

duckdb.sql(f"""
CREATE OR REPLACE TABLE faculty_salaries AS
{bucket_sql}
""")

In [127]:
pl.Config.set_tbl_rows(11)

# How many departments are accounted for in each bucket
duckdb.sql("""
SELECT COUNT(DISTINCT department) AS count, dep_bucket
FROM faculty_salaries
GROUP BY dep_bucket
ORDER BY count DESC
           """).pl()

count,dep_bucket
i64,str
209,"""Other"""
188,"""Humanities & Languages"""
182,"""Social Sciences & Communicatio…"
84,"""Education"""
82,"""Science & Mathematics"""
70,"""Business, Economics & Finance"""
68,"""Engineering"""
52,"""Computer Science"""
31,"""Architecture & Environmental D…"


Condense all titles to just tenure track or lecturer

In [129]:
final_df = duckdb.sql("""
SELECT DISTINCT
    year_transparentCA AS year,
    school AS campus,
    name,
    title AS title_scrapped,
    job AS title_tc,
    role,
    department_scraped AS original_department,
    dep_bucket AS department,
    totalpay AS gross_pay,
    base AS base_pay,
    overtime AS overtime_pay,
    other AS other_pay
FROM faculty_salaries
""").pl()

In [130]:
import polars as pl

def classify_title(column):
    title = (
        pl.col(column)
        .cast(pl.String)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[_\-/]+", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )

    return (
        pl.when(title.str.contains(r"\b(lecturer|visiting faculty)\b"))
        .then(pl.lit("lecturer"))
        .when(
            title.str.contains(
                r"\b(instructional faculty|isntructional faculty|"
                r"ft faculty unit|department chair)\b"
            )
        )
        .then(pl.lit("tenure_track"))
        .otherwise(pl.lit(None, dtype=pl.String))
    )

final_df = final_df.with_columns(
    pl.coalesce([
        classify_title("title_tc"),
        classify_title("title_scrapped"),
    ]).alias("final_title")
)

In [131]:
rows_before = final_df.height

final_df = final_df.filter(
    ~pl.col("title_tc")
    .fill_null("")
    .str.contains(r"(?i)\b(librarians?|research fello\w*|coaching|coaches|\w*intern\w*|student servic\w*|coach)\b")
)

print(f"Dropped {rows_before - final_df.height:,} rows")
print(f"Remaining: {final_df.height:,} rows")

Dropped 1,873 rows
Remaining: 158,751 rows


Drop supplemental pay roles which are defined as:
 ("supplemental_teaching_pay", r"special programs|summer session|extension|additional employment|music studio"),

In [132]:
# Drop supplemental pay; leave the source file unchanged.
supplemental_pay = final_df["role"].eq("supplemental_teaching_pay").fill_null(False)
dropped_supplemental_rows = int(supplemental_pay.sum())
final_df = final_df.filter(~supplemental_pay)
print(f"Dropped {dropped_supplemental_rows:,} supplemental teaching pay rows; "
      f"retained {final_df.height:,} CSU rows.")

Dropped 44,579 supplemental teaching pay rows; retained 114,172 CSU rows.


In [133]:
# Overall matching counts
display(
    final_df.select(
        pl.len().alias("total_rows"),
        pl.col("final_title").is_not_null().sum().alias("matched_rows"),
        pl.col("final_title").is_null().sum().alias("unmatched_rows"),
    )
)

# Counts by classification (null means unmatched)
display(
    final_df.group_by("final_title")
    .len(name="rows")
    .sort("rows", descending=True)
)

# Every unmatched title combination, most frequent first
unmatched_titles = (
    final_df.filter(pl.col("final_title").is_null())
    .group_by(["title_tc", "title_scrapped"])
    .len(name="rows")
    .sort("rows", descending=True)
)

print("Unmatched title combinations:")
with pl.Config(tbl_rows=-1, fmt_str_lengths=150):
    display(unmatched_titles)

# Sample matched rows for manual validation
for category in ["tenure_track", "lecturer"]:
    matches = final_df.filter(pl.col("final_title") == category)
    print(f"\nExamples: {category}")
    display(
        matches.select(
            "name", "campus", "title_tc", "title_scrapped", "final_title"
        ).sample(n=min(5, matches.height), seed=42)
    )

total_rows,matched_rows,unmatched_rows
u32,u32,u32
114172,114172,0


final_title,rows
str,u32
"""tenure_track""",78090
"""lecturer""",36082


Unmatched title combinations:


title_tc,title_scrapped,rows
str,str,u32



Examples: tenure_track


name,campus,title_tc,title_scrapped,final_title
str,str,str,str,str
"""Jessalyn Bolkema""","""California State University, D…","""instructional faculty - academ…","""Assistant Professor""","""tenure_track"""
"""Cathy Gaspard""","""CSU Northridge""","""instructional faculty - academ…","""Professor""","""tenure_track"""
"""Kah Lau""","""CSU Northridge""","""instructional faculty - academ…","""Professor""","""tenure_track"""
"""Rodrigo Gaitan""","""Sonoma State University""","""instructional faculty - academ…","""Instructional Faculty""","""tenure_track"""
"""Don Hankins""","""California State University, C…","""instructional faculty - academ…","""Professor""","""tenure_track"""



Examples: lecturer


name,campus,title_tc,title_scrapped,final_title
str,str,str,str,str
"""Christopher Roos""","""CSU Channel Islands""","""lecturer - academic year""","""Ext U Faculty""","""lecturer"""
"""Olav Smith""","""California State University, C…","""lecturer - academic year""","""Lecturer""","""lecturer"""
"""Eric Paton""","""Cal Poly San Luis Obispo""","""lecturer - academic year""","""""","""lecturer"""
"""Carol Mroz""","""CSU Los Angeles""","""lecturer - academic year""",null,"""lecturer"""
"""Ali Dehghan""","""CSU San Bernardino""","""lecturer - academic year""","""Lecturer""","""lecturer"""


In [134]:
final_df["role"].unique()

role
str
"""ladder_faculty"""
"""lecturer"""
"""clinical_adjunct_faculty"""


### Save the Cleaned Dataset for EDA
Save year, school, name, title (tenureck or lecturer), original department and dep_bucket, totalpay and basepay, overtime as overtime and other as other_pay
The output is sorted by school, name, year, bucket, and original department.

In [139]:
export_df = (
    final_df
    .sort([
        "campus", "name", "year", "role", "department",
        "original_department", "final_title", "gross_pay",
        "base_pay", "overtime_pay", "other_pay",
    ])
    .select(
        "year",
        "campus",
        "name",
        pl.col("final_title").alias("title"),
        "original_department",
        "department",
        "gross_pay",
        "base_pay",
        "overtime_pay",
        "other_pay",
    )
    .unique(maintain_order=True)
)

export_df.write_parquet("../data/processed/golden_csu_2013-2024.parquet")
export_df.describe()

statistic,year,campus,name,title,original_department,department,gross_pay,base_pay,overtime_pay,other_pay
str,f64,str,str,str,str,str,f64,f64,f64,f64
"""count""",111970.0,"""111970""","""111970""","""111970""","""109806""","""111970""",111970.0,111970.0,111970.0,111970.0
"""null_count""",0.0,"""0""","""0""","""0""","""2164""","""0""",0.0,0.0,0.0,0.0
"""mean""",2019.281424,null,null,null,null,null,75674.305698,68783.775833,1.141306,6889.210065
"""std""",3.348823,null,null,null,null,null,48491.658124,43406.950399,94.472145,11197.17854
"""min""",2013.0,"""CSU Bakersfield""","""Aakash Ahamed""","""lecturer""","""""","""Agriculture, Food & Animal""",0.0,0.0,0.0,-16941.0
"""25%""",2017.0,null,null,null,null,null,33333.0,29354.0,0.0,0.0
"""50%""",2020.0,null,null,null,null,null,80994.0,75645.0,0.0,1879.0
"""75%""",2022.0,null,null,null,null,null,108753.0,100455.0,0.0,9648.0
"""max""",2024.0,"""Sonoma State University""","""jody berger""","""tenure_track""","""the School of Journalism and M…","""Social Sciences & Communicatio…",373144.0,373144.0,16722.0,158025.0


In [138]:
duckdb.sql("""
SELECT COUNT(department) AS count, department, role, title_scrapped, title_tc
FROM final_df
GROUP BY department, role, title_scrapped, title_tc
LIMIT 200
           """).pl()

count,department,role,title_scrapped,title_tc
i64,str,str,str,str
572,"""Architecture & Environmental D…","""lecturer""","""Lecturer""","""lecturer - academic year"""
229,"""Education""","""ladder_faculty""",null,"""instructional faculty - academ…"
1778,"""Business, Economics & Finance""","""lecturer""","""Lecturer""","""lecturer - academic year"""
63,"""Engineering""","""lecturer""","""Assistant Professor""","""lecturer - academic year"""
18,"""Humanities & Languages""","""ladder_faculty""","""Professor Emeritus""","""department chair - academic ye…"
209,"""Science & Mathematics""","""ladder_faculty""","""Professor""","""department chair - academic ye…"
…,…,…,…,…
1,"""Education""","""ladder_faculty""","""Emeritus""","""instructional faculty - 12 mon…"
1,"""Humanities & Languages""","""lecturer""","""Lecturer of French & Italian""","""lecturer - 12 month"""
